# Cost Estimation & Budget Planning

Quantum hardware is expensive. This notebook shows how to use
`CostEstimator` and `BudgetOptimizer` to estimate costs, compare
backends, and plan experiments within a fixed budget.

We cover:
- `CostEstimator` basic usage and per-shot pricing
- Multi-backend cost comparison with `compare_backends()`
- Budget planning with `max_shots_within_budget()`
- `BudgetOptimizer` for automated backend + strategy selection
- `VENDOR_PRICING` exploration (all 9 backends)
- Cost scaling analysis and budget-constrained experiment planning

In [1]:
from qb_compiler.cost.estimator import CostEstimator, CostEstimate
from qb_compiler.cost.budget_optimizer import BudgetOptimizer
from qb_compiler.cost.pricing import VENDOR_PRICING, VendorPricing, cost_per_shot
from qb_compiler.config import BACKEND_CONFIGS
from qb_compiler.exceptions import BudgetExceededError, BackendNotSupportedError

## 1. Exploring VENDOR_PRICING

All 9 supported backends have pricing data. Costs vary by ~50,000x
between the cheapest (IBM) and most expensive (Quantinuum).

In [2]:
print(f"{'Backend':<18} {'Provider':<12} {'$/shot':>10} {'Notes'}")
print("-" * 72)

for name, pricing in sorted(VENDOR_PRICING.items(), key=lambda x: x[1].cost_per_shot_usd):
    print(
        f"{pricing.backend:<18} {pricing.provider:<12} "
        f"${pricing.cost_per_shot_usd:>9.5f} {pricing.notes}"
    )

cheapest = min(VENDOR_PRICING.values(), key=lambda p: p.cost_per_shot_usd)
priciest = max(VENDOR_PRICING.values(), key=lambda p: p.cost_per_shot_usd)
ratio = priciest.cost_per_shot_usd / cheapest.cost_per_shot_usd
print(f"\nPrice range: {ratio:,.0f}x between {cheapest.backend} and {priciest.backend}")

Backend            Provider         $/shot Notes
------------------------------------------------------------------------
ibm_torino         ibm          $  0.00014 Heron r1, 133q, Utility tier
ibm_fez            ibm          $  0.00016 Heron r2, 156q, Utility tier
ibm_marrakesh      ibm          $  0.00016 Heron r2, 156q, Utility tier
iqm_emerald        iqm          $  0.00020 Emerald, 5q, Braket pricing
rigetti_ankaa      rigetti      $  0.00035 Ankaa-3, 84q, Braket pricing
iqm_garnet         iqm          $  0.00045 Garnet, 20q, Braket pricing
ionq_aria          ionq         $  0.03000 Aria-2, 25q, Braket per-shot + per-task
ionq_forte         ionq         $  0.08000 Forte-1, 36q, Braket pricing
quantinuum_h2      quantinuum   $  8.00000 H2, 56q, HQC-based pricing (approximate per-shot)

Price range: 57,143x between ibm_torino and quantinuum_h2


## 2. CostEstimator Basic Usage

The simplest usage: estimate cost for a specific backend and shot count.

In [3]:
estimator = CostEstimator()

# How much does 10,000 shots on IBM Fez cost?
est = estimator.estimate("ibm_fez", shots=10_000)
print(f"Backend: {est.backend}")
print(f"Shots:   {est.shots:,}")
print(f"$/shot:  ${est.cost_per_shot:.5f}")
print(f"Total:   ${est.total_cost_usd:.2f}")
print(f"repr:    {est}")

Backend: ibm_fez
Shots:   10,000
$/shot:  $0.00016
Total:   $1.60
repr:    CostEstimate(backend='ibm_fez', shots=10,000, total=$1.6000)


In [4]:
# Same shot count, very different backends
shots = 1_000
for backend in ["ibm_fez", "rigetti_ankaa", "ionq_aria", "quantinuum_h2"]:
    est = estimator.estimate(backend, shots)
    print(f"{backend:<18} {shots:>6,} shots = ${est.total_cost_usd:>10.2f}")

ibm_fez             1,000 shots = $      0.16
rigetti_ankaa       1,000 shots = $      0.35
ionq_aria           1,000 shots = $     30.00
quantinuum_h2       1,000 shots = $   8000.00


## 3. Budget Enforcement

`CostEstimator` can enforce a budget cap. If an estimate exceeds it,
`BudgetExceededError` is raised -- catching runaway costs before submission.

In [5]:
# Create an estimator with a $50 budget
capped = CostEstimator(budget_usd=50.0)

# This is fine
est = capped.estimate("ibm_fez", shots=100_000)
print(f"IBM Fez, 100K shots: ${est.total_cost_usd:.2f} -- within budget")

# This will raise
try:
    capped.estimate("ionq_aria", shots=1_000)
except BudgetExceededError as e:
    print(f"\nBudget exceeded: {e}")
    print(f"  Estimated: ${e.estimated_usd:.2f}")
    print(f"  Budget:    ${e.budget_usd:.2f}")

IBM Fez, 100K shots: $16.00 -- within budget


## 4. Multi-Backend Cost Comparison

`compare_backends()` estimates costs across multiple (or all) backends
and returns them sorted cheapest-first.

In [6]:
estimator = CostEstimator()

# Compare all backends for 10,000 shots
comparison = estimator.compare_backends(shots=10_000)

print(f"Cost comparison for 10,000 shots (sorted cheapest first):")
print(f"{'Rank':>4} {'Backend':<18} {'$/shot':>10} {'Total':>10}")
print("-" * 44)
for i, est in enumerate(comparison, 1):
    print(f"{i:>4} {est.backend:<18} ${est.cost_per_shot:>9.5f} ${est.total_cost_usd:>9.2f}")

Cost comparison for 10,000 shots (sorted cheapest first):
Rank Backend                $/shot      Total
--------------------------------------------
   1 ibm_torino         $  0.00014 $     1.40
   2 ibm_fez            $  0.00016 $     1.60
   3 ibm_marrakesh      $  0.00016 $     1.60
   4 iqm_emerald        $  0.00020 $     2.00
   5 rigetti_ankaa      $  0.00035 $     3.50
   6 iqm_garnet         $  0.00045 $     4.50
   7 ionq_aria          $  0.03000 $   300.00
   8 ionq_forte         $  0.08000 $   800.00
   9 quantinuum_h2      $  8.00000 $ 80000.00


In [7]:
# Compare only a subset of backends
superconducting = estimator.compare_backends(
    shots=50_000,
    backends=["ibm_fez", "ibm_torino", "ibm_marrakesh", "rigetti_ankaa", "iqm_garnet"],
)

print("Superconducting backends only (50K shots):")
for est in superconducting:
    print(f"  {est.backend:<18} ${est.total_cost_usd:>8.2f}")

Superconducting backends only (50K shots):
  ibm_torino         $    7.00
  ibm_fez            $    8.00
  ibm_marrakesh      $    8.00
  rigetti_ankaa      $   17.50
  iqm_garnet         $   22.50


## 5. Budget Planning: max_shots_within_budget()

Given a fixed budget, how many shots can you afford on each backend?

In [8]:
estimator = CostEstimator()
budget = 100.0  # $100 budget

print(f"Maximum shots within ${budget:.0f} budget:")
print(f"{'Backend':<18} {'Max shots':>12} {'Actual cost':>12}")
print("-" * 44)

for name in sorted(VENDOR_PRICING.keys()):
    max_shots = estimator.max_shots_within_budget(name, budget_usd=budget)
    actual_cost = max_shots * cost_per_shot(name)
    print(f"{name:<18} {max_shots:>12,} ${actual_cost:>11.2f}")

Maximum shots within $100 budget:
Backend               Max shots  Actual cost
--------------------------------------------
ibm_fez                 625,000 $     100.00
ibm_marrakesh           625,000 $     100.00
ibm_torino              714,285 $     100.00
ionq_aria                 3,333 $      99.99
ionq_forte                1,250 $     100.00
iqm_emerald             500,000 $     100.00
iqm_garnet              222,222 $     100.00
quantinuum_h2                12 $      96.00
rigetti_ankaa           285,714 $     100.00


## 6. BudgetOptimizer: Finding Optimal Settings

`BudgetOptimizer` goes beyond cost estimation -- it recommends the
best compilation strategy and shot count for a given budget.

In [9]:
optimizer = BudgetOptimizer(min_shots=100)

# Optimize for IBM Fez with a $10 budget
result = optimizer.optimize("ibm_fez", budget_usd=10.0, circuit_depth=50)
print(f"Optimization for IBM Fez ($10 budget):")
print(f"  Recommended shots:  {result.recommended_shots:,}")
print(f"  Estimated fidelity: {result.estimated_fidelity:.4f}")
print(f"  Estimated cost:     ${result.estimated_cost_usd:.2f}")
print(f"  Strategy:           {result.strategy}")
print(f"  Notes:              {result.notes}")

Optimization for IBM Fez ($10 budget):
  Recommended shots:  62,499
  Estimated fidelity: 0.7705
  Estimated cost:     $10.00
  Strategy:           speed_optimal
  Notes:              Within budget


In [10]:
# The optimizer chooses different strategies based on per-shot cost:
# - Expensive backends (>= $0.10/shot): fidelity_optimal
# - Mid-range ($0.001 - $0.10/shot): cost_optimal
# - Cheap (< $0.001/shot): speed_optimal

print(f"{'Backend':<18} {'$/shot':>10} {'Strategy':<18} {'Notes'}")
print("-" * 72)
for name in sorted(VENDOR_PRICING.keys()):
    try:
        r = optimizer.optimize(name, budget_usd=50.0)
        cps = VENDOR_PRICING[name].cost_per_shot_usd
        print(f"{name:<18} ${cps:>9.5f} {r.strategy:<18} {r.notes}")
    except (BudgetExceededError, ValueError) as e:
        print(f"{name:<18} -- Budget exceeded")

Backend                $/shot Strategy           Notes
------------------------------------------------------------------------
ibm_fez            $  0.00016 speed_optimal      Within budget
ibm_marrakesh      $  0.00016 speed_optimal      Within budget
ibm_torino         $  0.00014 speed_optimal      Within budget
ionq_aria          $  0.03000 cost_optimal       Within budget
ionq_forte         $  0.08000 cost_optimal       Within budget
iqm_emerald        $  0.00020 speed_optimal      Within budget
iqm_garnet         $  0.00045 speed_optimal      Within budget
quantinuum_h2      -- Budget exceeded
rigetti_ankaa      $  0.00035 speed_optimal      Within budget


## 7. Finding the Cheapest Backend Automatically

`find_cheapest_backend()` scans all backends and returns the one with
the best fidelity-to-cost ratio within your budget.

In [11]:
optimizer = BudgetOptimizer(min_shots=100)

# Find cheapest backend for 1,000 shots within $5
best = optimizer.find_cheapest_backend(
    budget_usd=5.0,
    target_shots=1_000,
    min_qubits=5,
)

if best:
    print(f"Best option for 1,000 shots within $5 (>= 5 qubits):")
    print(f"  Backend:  {best.backend}")
    print(f"  Cost:     ${best.estimated_cost_usd:.2f}")
    print(f"  Fidelity: {best.estimated_fidelity:.4f}")
    print(f"  Strategy: {best.strategy}")
else:
    print("No backend fits the budget.")

Best option for 1,000 shots within $5 (>= 5 qubits):
  Backend:  ibm_fez
  Cost:     $0.16
  Fidelity: 0.7705
  Strategy: speed_optimal


In [12]:
# What if we need more qubits?
for min_q in [5, 20, 50, 100]:
    best = optimizer.find_cheapest_backend(
        budget_usd=100.0,
        target_shots=10_000,
        min_qubits=min_q,
    )
    if best:
        print(
            f">= {min_q:>3} qubits: {best.backend:<18} "
            f"${best.estimated_cost_usd:>8.2f}  "
            f"fidelity={best.estimated_fidelity:.3f}"
        )
    else:
        print(f">= {min_q:>3} qubits: no backend fits $100 budget")

>=   5 qubits: ibm_fez            $    1.60  fidelity=0.771
>=  20 qubits: ibm_fez            $    1.60  fidelity=0.771
>=  50 qubits: ibm_fez            $    1.60  fidelity=0.771
>= 100 qubits: ibm_fez            $    1.60  fidelity=0.771


## 8. Cost Scaling Analysis

How does cost scale with shot count? Linear for all backends,
but the slope varies dramatically.

In [13]:
estimator = CostEstimator()
shot_counts = [100, 1_000, 10_000, 100_000, 1_000_000]

# Pick representative backends from each price tier
backends = ["ibm_torino", "iqm_garnet", "rigetti_ankaa", "ionq_aria", "quantinuum_h2"]

print(f"{'Shots':>10}", end="")
for b in backends:
    print(f" | {b:>14}", end="")
print()
print("-" * 90)

for shots in shot_counts:
    print(f"{shots:>10,}", end="")
    for b in backends:
        est = estimator.estimate(b, shots)
        if est.total_cost_usd >= 1000:
            print(f" | ${est.total_cost_usd:>12,.0f}", end="")
        elif est.total_cost_usd >= 1:
            print(f" | ${est.total_cost_usd:>12,.2f}", end="")
        else:
            print(f" | ${est.total_cost_usd:>12,.4f}", end="")
    print()

     Shots |     ibm_torino |     iqm_garnet |  rigetti_ankaa |      ionq_aria |  quantinuum_h2
------------------------------------------------------------------------------------------
       100 | $      0.0140 | $      0.0450 | $      0.0350 | $        3.00 | $      800.00
     1,000 | $      0.1400 | $      0.4500 | $      0.3500 | $       30.00 | $       8,000
    10,000 | $        1.40 | $        4.50 | $        3.50 | $      300.00 | $      80,000
   100,000 | $       14.00 | $       45.00 | $       35.00 | $       3,000 | $     800,000
 1,000,000 | $      140.00 | $      450.00 | $      350.00 | $      30,000 | $   8,000,000


## 9. Budget-Constrained Experiment Planning

Real experiments involve multiple circuits. Given a fixed budget,
how should you allocate shots across circuits?

In [14]:
def plan_experiment(backend, budget_usd, n_circuits, min_shots_per_circuit=100):
    """Plan an experiment: distribute budget evenly across circuits."""
    cps = cost_per_shot(backend)
    total_shots = int(budget_usd / cps) if cps > 0 else 1_000_000
    shots_per_circuit = total_shots // n_circuits

    if shots_per_circuit < min_shots_per_circuit:
        feasible_circuits = total_shots // min_shots_per_circuit
        return {
            "feasible": False,
            "reason": f"Only {feasible_circuits} circuits feasible at {min_shots_per_circuit} shots/circuit",
            "shots_per_circuit": min_shots_per_circuit,
            "max_circuits": feasible_circuits,
            "total_cost": feasible_circuits * min_shots_per_circuit * cps,
        }

    return {
        "feasible": True,
        "n_circuits": n_circuits,
        "shots_per_circuit": shots_per_circuit,
        "total_shots": shots_per_circuit * n_circuits,
        "total_cost": shots_per_circuit * n_circuits * cps,
        "budget_remaining": budget_usd - (shots_per_circuit * n_circuits * cps),
    }


# Scenario: $200 budget, 20 VQE circuits on different backends
budget = 200.0
n_circuits = 20

print(f"Experiment plan: {n_circuits} circuits, ${budget:.0f} budget")
print(f"{'Backend':<18} {'Feasible':>8} {'Shots/circ':>12} {'Total cost':>12}")
print("-" * 52)

for backend in ["ibm_fez", "rigetti_ankaa", "iqm_garnet", "ionq_aria", "quantinuum_h2"]:
    plan = plan_experiment(backend, budget, n_circuits)
    if plan["feasible"]:
        print(
            f"{backend:<18} {'yes':>8} {plan['shots_per_circuit']:>12,} "
            f"${plan['total_cost']:>11.2f}"
        )
    else:
        print(
            f"{backend:<18} {'NO':>8}  -- {plan['reason']}"
        )

Experiment plan: 20 circuits, $200 budget
Backend            Feasible   Shots/circ   Total cost
----------------------------------------------------
ibm_fez                 yes       62,500 $     200.00
rigetti_ankaa           yes       28,571 $     200.00
iqm_garnet              yes       22,222 $     200.00
ionq_aria               yes          333 $     199.80
quantinuum_h2            NO  -- Only 0 circuits feasible at 100 shots/circuit


## 10. Building a Cost Calculator Utility

Combining `CostEstimator` and `BudgetOptimizer` into a reusable
planning function for day-to-day use.

In [15]:
def cost_report(budget_usd, target_shots, circuit_depth=50):
    """Generate a comprehensive cost report for a planned experiment."""
    estimator = CostEstimator()
    optimizer = BudgetOptimizer(min_shots=100)

    print(f"=" * 60)
    print(f"COST REPORT")
    print(f"Budget: ${budget_usd:.2f} | Target shots: {target_shots:,}")
    print(f"=" * 60)

    # Cost comparison
    print(f"\n--- Cost by Backend ---")
    comparison = estimator.compare_backends(shots=target_shots)
    for est in comparison:
        within = "OK" if est.total_cost_usd <= budget_usd else "OVER"
        print(f"  {est.backend:<18} ${est.total_cost_usd:>10.2f}  [{within}]")

    # Best option
    print(f"\n--- Recommended ---")
    best = optimizer.find_cheapest_backend(
        budget_usd=budget_usd,
        target_shots=target_shots,
    )
    if best:
        print(f"  Backend:  {best.backend}")
        print(f"  Shots:    {best.recommended_shots:,}")
        print(f"  Cost:     ${best.estimated_cost_usd:.2f}")
        print(f"  Fidelity: {best.estimated_fidelity:.4f}")
        print(f"  Strategy: {best.strategy}")
    else:
        print("  No backend fits the budget for the requested shot count.")
        print("  Consider reducing target_shots or increasing budget.")

    print(f"\n" + "=" * 60)


# Example: $50 budget, 10,000 shots
cost_report(budget_usd=50.0, target_shots=10_000)

COST REPORT
Budget: $50.00 | Target shots: 10,000

--- Cost by Backend ---
  ibm_torino         $      1.40  [OK]
  ibm_fez            $      1.60  [OK]
  ibm_marrakesh      $      1.60  [OK]
  iqm_emerald        $      2.00  [OK]
  rigetti_ankaa      $      3.50  [OK]
  iqm_garnet         $      4.50  [OK]
  ionq_aria          $    300.00  [OVER]
  ionq_forte         $    800.00  [OVER]
  quantinuum_h2      $  80000.00  [OVER]

--- Recommended ---
  Backend:  ibm_fez
  Shots:    10,000
  Cost:     $1.60
  Fidelity: 0.7705
  Strategy: speed_optimal



## 11. Error Handling

The cost estimation API raises clear exceptions for common mistakes.

In [16]:
estimator = CostEstimator()

# Unknown backend
try:
    estimator.estimate("not_a_real_backend", 1000)
except BackendNotSupportedError as e:
    print(f"BackendNotSupportedError: {e}")

# Negative shots
try:
    estimator.estimate("ibm_fez", -100)
except ValueError as e:
    print(f"ValueError: {e}")

# Budget optimizer with impossible constraints
optimizer = BudgetOptimizer(min_shots=100)
try:
    optimizer.optimize("quantinuum_h2", budget_usd=1.0)
except BudgetExceededError as e:
    print(f"BudgetExceededError: cannot afford minimum shots on Quantinuum")

BackendNotSupportedError: Backend 'not_a_real_backend' is not supported. Available: ibm_fez, ibm_torino, ibm_marrakesh, ionq_aria, ionq_forte, iqm_garnet, iqm_emerald, rigetti_ankaa, quantinuum_h2
ValueError: shots must be non-negative, got -100
BudgetExceededError: cannot afford minimum shots on Quantinuum


## Summary

| Class | Purpose | Key methods |
|-------|---------|-------------|
| `CostEstimator` | Per-backend cost calculation | `estimate()`, `compare_backends()`, `max_shots_within_budget()` |
| `BudgetOptimizer` | Budget-aware backend + strategy selection | `optimize()`, `find_cheapest_backend()` |
| `VENDOR_PRICING` | Raw pricing data for all 9 backends | Dict lookup, `cost_per_shot()` |

**Key insight:** Cost and fidelity are inversely correlated. The cheapest
backends (IBM, IQM) have higher error rates, while the most expensive
(Quantinuum, IonQ) have the lowest. `BudgetOptimizer` navigates this
trade-off automatically.